# Segunda Parte del Proyecto
## Procesamiento de Datos a Gran Escala — Entrega 2
**Pontificia Universidad Javeriana | Departamento de Ingeniería de Sistemas**

Equipo de consultoría contratado por el Ministerio de Educación para analizar los resultados ICFES 11 en relación con el servicio de internet por municipio del departamento de Boyacá.

Esta segunda entrega abarca:
1. Filtros y transformaciones finales
2. Respuesta a las preguntas de negocio planteadas
3. Selección y justificación de técnicas de ML
4. Preparación de datos para modelado
5. Aplicación de modelos supervisado (Random Forest) y no supervisado (K-Means) con MLlib
6. Evaluación con métricas apropiadas


## 1. Carga de Datasets

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, upper, trim, translate, when, avg, count, sum as spark_sum
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, ClusteringEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [0]:
# ============================
# UTILIDADES GENERALES
# ============================

import pyspark.sql.functions as F
from pyspark.sql.functions import col, upper, trim, translate

# 1. Estandarización de nombres de municipio
def std_municipio(df, col_name):
    """
    Normaliza los nombres de municipios:
    - Mayúsculas
    - Sin tildes
    - Reemplaza Ñ por N
    Guarda el resultado en la columna 'municipio_std'.
    """
    df = df.withColumn("municipio_std", upper(trim(F.col(col_name))))
    df = df.withColumn(
        "municipio_std",
        translate(F.col("municipio_std"), "ÁÉÍÓÚÄËÏÖÜÑ", "AEIOUAEIOUN")
    )
    return df


# 2. Cast seguro y genérico (evita CAST_INVALID_INPUT)
def cast_safe(df, col_in, col_out, tipo="double", regex=None):
    """
    Castea una columna de forma robusta:
    - Trata 'null', '' y espacios como NULL reales
    - Opcionalmente aplica un regex (por ejemplo para quitar % y comas)
    - Devuelve un nuevo DF con la columna col_out ya casteada
    """
    c_str = F.col(col_in).cast("string")

    # Normalización de nulos y vacíos
    base_expr = F.when(
        c_str.isNull() |
        (F.trim(c_str) == "") |
        (F.trim(c_str) == "null"),
        None
    )

    # Si hay regex, se limpia antes de castear
    if regex is not None:
        expr = base_expr.otherwise(
            F.regexp_replace(c_str, regex, "").cast(tipo)
        )
    else:
        expr = base_expr.otherwise(c_str.cast(tipo))

    return df.withColumn(col_out, expr)


# 3. Limpieza específica de porcentajes (98,71%, 6,25%, etc.)
def clean_pct_safe(df, colname):
    """
    Limpia una columna de porcentaje que viene como string con formato '98,71%' o similar.
    Crea una nueva columna <colname>_num en double sin '%', comas ni problemas de cast.
    """
    return cast_safe(
        df,
        col_in=colname,
        col_out=colname + "_num",
        tipo="double",
        regex="[%,]"   # elimina % y comas
    )


In [0]:
# Instituciones Oficiales con Internet
dfInternetOficiales = spark.read.csv(
    "/Volumes/workspace/proyecto/proyectopdd/1__Instituciones_Oficiales_Internet.csv",
    header=True, inferSchema=True
)
dfInternetOficiales.createOrReplaceTempView("dfInternetOficiales")
print(f"Registros: {dfInternetOficiales.count()} | Columnas: {len(dfInternetOficiales.columns)}")
display(dfInternetOficiales.limit(5))

Registros: 1914 | Columnas: 18


provincia,codigo-municipio,municipio,codigo-dane-institucion-educativa,institucion-educativa,codigo-dane-escuela,escuela,zona,proyectos-de-conectividad-2024,operador,estado,medio-de-enlace,ancho-de-banda-de-subida,ancho-de-banda-descarga,finalizacion-del-contrato,latitud,longitud,tiene_ancho_banda
ORIENTE,"15,022",ALMEIDA,"315,022,000,196",I.E. ENRIQUE SUAREZ,"215,022,000,078",ESC EL ROSAL,RURAL,CENTRO DIGITAL,UNIÓN TEMPORAL DE ETB NET COLOMBIA CONECTADA,EN OPERACIÓN,SIN DATOS,3,12,2032 Dec 01 12:00:00 AM,4.94506202,-73.38084868,1
SUGAMUXI,"15,047",AQUITANIA,"115,047,000,019",I.E. TECNICA RAMON IGNACIO AVELLA,"215,047,000,196",ESC CAJON,RURAL,CENTRO DIGITAL,UNIÓN TEMPORAL DE ETB NET COLOMBIA CONECTADA,PENDIENTE INICIO OPERACION,SIN DATOS,3,12,2032 Dec 01 12:00:00 AM,5.369444,-72.881389,1
SUGAMUXI,"15,047",AQUITANIA,"115,047,000,019",I.E. TECNICA RAMON IGNACIO AVELLA,"215,047,000,439",ESC DAITO,RURAL,CENTRO DIGITAL,UNIÓN TEMPORAL DE ETB NET COLOMBIA CONECTADA,EN OPERACIÓN,SIN DATOS,3.75,15,2032 Dec 01 12:00:00 AM,5.498777603,-72.91264973,1
SUGAMUXI,"15,047",AQUITANIA,"215,047,000,099",I.E. REGION SUR DE AQUITANIA,"215,047,000,579",ESC DIGANOME,RURAL,CENTRO DIGITAL,UNIÓN TEMPORAL DE ETB NET COLOMBIA CONECTADA,PENDIENTE INICIO OPERACION,SIN DATOS,3,12,2032 Dec 01 12:00:00 AM,5.304167,-72.927778,1
SUGAMUXI,"15,047",AQUITANIA,"115,047,000,019",I.E. TECNICA RAMON IGNACIO AVELLA,"215,047,000,188",ESC EL TOBAL,RURAL,CENTRO DIGITAL,UNIÓN TEMPORAL DE ETB NET COLOMBIA CONECTADA,PENDIENTE INICIO OPERACION,SIN DATOS,3,12,2032 Dec 01 12:00:00 AM,5.508333,-72.869167,1


In [0]:
# Instituciones No Oficiales con Internet
dfInternetNoOficiales = spark.read.csv(
    "/Volumes/workspace/proyecto/proyectopdd/2__Instituciones_No_Oficiales_Internet.csv",
    header=True, inferSchema=True
)
dfInternetNoOficiales.createOrReplaceTempView("dfInternetNoOficiales")
print(f"Registros: {dfInternetNoOficiales.count()} | Columnas: {len(dfInternetNoOficiales.columns)}")
display(dfInternetNoOficiales.limit(5))

Registros: 409 | Columnas: 14


codigo-dane-institucion-educativa,institucion-educativa,codigo-dane-institucion-escuela,escuela,municipio,longitud,latitud,tipo-sede,zona,tecnologia-conexion,programa-conexion,anio,ancho-de-banda-num,tiene-ancho-banda
215.531.000.125,I.E. SANTA ROSA,215.531.000.478,I.E. SANTA ROSA - SEDE PRINCIPAL,PAUNA,"-74,009265","5,70746",SEDE PRINCIPAL,RURAL,SIN DATOS,CENTROS DIGITALES MINTIC,2.023,null,0
215.022.000.078,I.E. ENRIQUE SUAREZ,315.022.000.196,ESC EL ROSAL,ALMEIDA,"-73,380849","4,945062",SEDE NO PRINCIPAL,RURAL,SIN DATOS,CENTROS DIGITALES MINTIC,2.023,null,0
215.087.000.253,I.E. TECNICA SUSANA GUILLEMIN,115.087.000.011,ESC ALTO DE CANUTOS,BELÉN,"-72,923567","6,038845",SEDE NO PRINCIPAL,RURAL,SIN DATOS,CENTROS DIGITALES MINTIC,2.023,null,0
215.106.000.146,I.E. TECNICA MANUEL BRICEÑO,315.106.000.124,COL BAS MEDIA LUNA,BRICEÑO,"-73,917392","5,706757",SEDE NO PRINCIPAL,RURAL,SIN DATOS,CENTROS DIGITALES MINTIC,2.023,null,0
215.109.000.163,I.E. TECNICA AGROPECUARIA LA GRANJA,215.109.000.163,SEDE PRINCIPAL,BUENAVISTA,"-73,984046","5,485597",SEDE PRINCIPAL,RURAL,SIN DATOS,CENTROS DIGITALES MINTIC,2.023,null,0


In [0]:
# Matrícula por Municipio
dfMatricula = spark.read.csv(
    "/Volumes/workspace/proyecto/proyectopdd/3__Matrícula_Por_Municipio.csv",
    header=True, inferSchema=True
)
dfMatricula.createOrReplaceTempView("dfMatricula")
print(f"Registros: {dfMatricula.count()} | Columnas: {len(dfMatricula.columns)}")
display(dfMatricula.limit(5))

Registros: 1368 | Columnas: 11


AÑO,CODIGO_MUNICIPIO,MUNICIPIO,TECNICA,TECNOLOGICA,UNIVERSITARIA,ESPECIALIZACION,MAESTRIA,DOCTORADO,CANTIDAD_UNIVERSIDADES,TOTAL_ESTUDIANTES
2005,15001,TUNJA,3400,3171,20021,4910,1420,34,11,32956
2005,15176,CHIQUINQUIRÁ,760,240,6940,0,0,0,4,7940
2005,15187,CHIVATÁ,340,0,0,0,0,0,1,340
2005,15238,DUITAMA,340,340,2817,470,0,0,5,3967
2005,15272,FIRAVITOBA,370,0,0,0,0,0,1,370


In [0]:
# Estadística Básica Media (coberturas, deserción, aprobación)
dfEstadisticas = spark.read.csv(
    "/Volumes/workspace/proyecto/proyectopdd/4__Estadística_Básica_Media.csv",
    header=True, inferSchema=True
)
dfEstadisticas.createOrReplaceTempView("dfEstadisticas")
print(f"Registros: {dfEstadisticas.count()} | Columnas: {len(dfEstadisticas.columns)}")
display(dfEstadisticas.limit(5))

Registros: 1722 | Columnas: 41


AÑO,CÓDIGO_MUNICIPIO,MUNICIPIO,CÓDIGO_DEPARTAMENTO,DEPARTAMENTO,CÓDIGO_ETC,ETC,POBLACIÓN_5_16,TASA_MATRICULACIÓN_5_16,COBERTURA_NETA,COBERTURA_NETA_TRANSICIÓN,COBERTURA_NETA_PRIMARIA,COBERTURA_NETA_SECUNDARIA,COBERTURA_NETA_MEDIA,COBERTURA_BRUTA,COBERTURA_BRUTA_TRANSICIÓN,COBERTURA_BRUTA_PRIMARIA,COBERTURA_BRUTA_SECUNDARIA,COBERTURA_BRUTA_MEDIA,TAMAÑO_PROMEDIO_DE_GRUPO,SEDES_CONECTADAS_A_INTERNET,DESERCIÓN,DESERCIÓN_TRANSICIÓN,DESERCIÓN_PRIMARIA,DESERCIÓN_SECUNDARIA,DESERCIÓN_MEDIA,APROBACIÓN,APROBACIÓN_TRANSICIÓN,APROBACIÓN_PRIMARIA,APROBACIÓN_SECUNDARIA,APROBACIÓN_MEDIA,REPROBACIÓN,REPROBACIÓN_TRANSICIÓN,REPROBACIÓN_PRIMARIA,REPROBACIÓN_SECUNDARIA,REPROBACIÓN_MEDIA,REPITENCIA,REPITENCIA_TRANSICIÓN,REPITENCIA_PRIMARIA,REPITENCIA_SECUNDARIA,REPITENCIA_MEDIA
2024,15491,Nobsa,15,Boyacá,3769.0,Boyacá (ETC),2989.0,"108,26%",108.13,"87,76%","113,55%","98,71%","60,8%",112.88,"104,64%","123,73%","117,01%","83,37%",null,null,"2,84%","1,69%","2,04%","4,08%",2.61,87.44,0%,"91,71%","79,31%","90,3%","9,72%",0%,"6,25%","16,61%","7,09%",7.1,"0,85%","4,08%","12,38%","5,6%"
2024,15215,Corrales,15,Boyacá,3769.0,Boyacá (ETC),416.0,"94,47%",94.47,"66,67%","90,12%","85,82%","48,65%",97.12,"86,11%","111,63%","102,24%","59,46%",null,null,"2,72%",0%,"4,17%","2,19%",0,91.34,0%,"89,58%","89,78%","97,73%","5,94%",0%,"6,25%","8,03%","2,27%",7.18,0%,"7,29%","10,22%","2,27%"
2024,15218,Covarachía,15,Boyacá,3769.0,Boyacá (ETC),531.0,"76,84%",76.84,"73,91%","73,66%","63,48%","53,01%",81.17,"76,09%","81,7%","82,58%","79,52%",null,null,"1,39%",0%,0%,"4,08%",0,96.29,0%,"98,91%","90,48%",100%,"2,32%",0%,"1,09%","5,44%",0%,4.64,0%,"3,28%","9,52%",0%
2024,15774,Susacón,15,Boyacá,3769.0,Boyacá (ETC),455.0,"59,34%",59.34,"45,71%","50,8%","61,29%","57,69%",61.54,"48,57%","55,61%","67,74%","69,23%",null,null,"4,64%",0%,"2,88%","2,86%",12.96,94.29,0%,"96,15%","97,14%","83,33%","1,07%",0%,"0,96%",0%,"3,7%",8.93,0%,"7,69%","11,43%","9,26%"
2024,15425,Macanal,15,Boyacá,3769.0,Boyacá (ETC),818.0,"64,55%",64.55,"53,13%","65,18%","58,99%",55%,68.09,"57,81%","68,15%","67,99%","72,86%",null,null,"2,51%",0%,"0,87%","4,76%",2.94,96.05,0%,"99,13%","91,01%","97,06%","1,44%",0%,0%,"4,23%",0%,7.36,"2,7%","2,62%","16,4%","2,94%"


In [0]:
# Estrategia UNIDOS (niveles de pobreza y logros sociales)
dfUNIDOS = spark.read.csv(
    "/Volumes/workspace/proyecto/proyectopdd/5__Estrategia_UNIDOS.csv",
    header=True, inferSchema=True
)
dfUNIDOS.createOrReplaceTempView("dfUNIDOS")
print(f"Registros: {dfUNIDOS.count()} | Columnas: {len(dfUNIDOS.columns)}")
display(dfUNIDOS.limit(5))

Registros: 50773 | Columnas: 69


CodigoFamilia,TipoDocumento,RangoEdad,CodigoMunicipioAtencion,NombreMunicipioAtencion,PuntajeSISBEN,Discapacidad,EstadoCivil,Estrato,Genero,Parentesco,TipoPoblacion,EstadoBeneficiario,Logro1,Logro2,Logro3,Logro4,Logro5,Logro6,Logro7,Logro8,Logro9,Logro10,Logro11,Logro12,Logro13,Logro14,Logro15,Logro16,Logro17,Logro18,Logro19,Logro20,Logro21,Logro22,Logro23,Logro24,Logro25,Logro26,PE34,PE35,PE36,PE37,PE38,PE39,PE40,PE41,PE42,PE43,PE44,PE45,PE46,PE48,PE50,PE51,HE11,HE12,HE13,HE14,HE15,HE16,HE17,HE18,HE20,HE21,HE22,HE23,total_logros_alcanzados,nivel_educativo_bajo
HID23735355,TI,18-29,15332,GÜICAN,4,NO,Sin Información,0,Hombre,HIJOS / HIJASTROS,UNIDOS RURAL,ACTIVO,ALCANZADO,ALCANZADO,NO APLICA,NO APLICA,NO APLICA,NO APLICA,POR ALCANZAR,POR ALCANZAR,ALCANZADO,ALCANZADO,NO APLICA,NO APLICA,NO APLICA,NO APLICA,POR ALCANZAR,ALCANZADO,NO APLICA,NO APLICA,ALCANZADO,NO APLICA,ALCANZADO,ALCANZADO,POR ALCANZAR,NO APLICA,NO APLICA,POR ALCANZAR,Si,No,No,No,No,Si,Si,No,BÁSICA PRIMARIA 5°,Si,No,Si,No,No,No,No,Si,No,Si,No,Si,Si,7,2,No,No,Si,No,8,1
HID54106531,CC,18-29,15051,ARCABUCO,1,NO,Sin Información,0,Mujer,JEFE,UNIDOS U100,ACTIVO,ALCANZADO,ALCANZADO,NO APLICA,NO APLICA,NO APLICA,NO APLICA,NO APLICA,NO APLICA,ALCANZADO,ALCANZADO,NO APLICA,NO APLICA,NO APLICA,NO APLICA,ALCANZADO,POR ALCANZAR,ALCANZADO,POR ALCANZAR,POR ALCANZAR,POR ALCANZAR,ALCANZADO,ALCANZADO,ALCANZADO,NO APLICA,POR ALCANZAR,NO APLICA,Si,No,No,No,No,No,Si,No,BÁSICA SECUNDARIA 7°,No,No,No,No,No,No,No,Si,No,Si,Si,Si,Si,4,2,No,No,No,No,9,0
HID41032193,CC,50-65,15224,CUCAITA,1,NO,Sin Información,1,Hombre,JEFE,UNIDOS RURAL,ACTIVO,ALCANZADO,ALCANZADO,NO APLICA,NO APLICA,NO APLICA,NO APLICA,NO APLICA,NO APLICA,ALCANZADO,ALCANZADO,NO APLICA,ALCANZADO,NO APLICA,NO APLICA,POR ALCANZAR,POR ALCANZAR,ALCANZADO,POR ALCANZAR,POR ALCANZAR,POR ALCANZAR,ALCANZADO,ALCANZADO,ALCANZADO,NO APLICA,ALCANZADO,POR ALCANZAR,Si,No,No,No,No,No,Si,No,BÁSICA PRIMARIA 5°,Si,No,No,No,No,No,No,Si,No,No,No,Si,Si,5,2,No,No,Si,No,10,1
HID40245288,RC,06-17,15774,SUSACON,1,NO,Sin Información,0,Mujer,HIJOS / HIJASTROS,UNIDOS RURAL,ACTIVO,ALCANZADO,ALCANZADO,ALCANZADO,NO APLICA,NO APLICA,ALCANZADO,NO APLICA,NO APLICA,POR ALCANZAR,ALCANZADO,NO APLICA,NO APLICA,ALCANZADO,ALCANZADO,POR ALCANZAR,NO APLICA,NO APLICA,NO APLICA,NO APLICA,NO APLICA,ALCANZADO,ALCANZADO,ALCANZADO,NO APLICA,NO APLICA,POR ALCANZAR,Si,Si,Si,Si,Si,No,No,No,NINGUNO,No,No,No,No,No,Si,Si,Si,No,Si,No,No,Si,5,2,No,No,Si,No,10,1
HID82948293,TI,18-29,15176,CHIQUINQUIRA,1,NO,Sin Información,0,Mujer,HIJOS/ HIJASTROS,UNIDOS U100,ACTIVO,ALCANZADO,POR ALCANZAR,NO APLICA,NO APLICA,NO APLICA,NO APLICA,ALCANZADO,ALCANZADO,ALCANZADO,ALCANZADO,NO APLICA,NO APLICA,NO APLICA,NO APLICA,ALCANZADO,ALCANZADO,NO APLICA,NO APLICA,ALCANZADO,NO APLICA,ALCANZADO,ALCANZADO,ALCANZADO,NO APLICA,NO APLICA,NO APLICA,No,No,No,No,No,Si,Si,Si,BÁSICA SECUNDARIA 6°,No,No,Si,No,No,No,No,Si,No,Si,Si,Si,Si,4,3,No,No,No,No,11,0


In [0]:
# Internet Fijo por Tecnología y Segmento
dfInternet = spark.read.csv(
    "/Volumes/workspace/proyecto/proyectopdd/6__Internet_Fijo_Tecnología_Segmento.csv",
    header=True, inferSchema=True
)
dfInternet.createOrReplaceTempView("dfInternet")
print(f"Registros: {dfInternet.count()} | Columnas: {len(dfInternet.columns)}")
display(dfInternet.limit(5))

Registros: 141561 | Columnas: 12


AÑO,TRIMESTRE,PROVEEDOR,COD_MUNICIPIO,MUNICIPIO,SEGMENTO,TECNOLOGIA,VELOCIDAD_BAJADA,VELOCIDAD_SUBIDA,No DE ACCESOS,TIPO_TECNOLOGIA,ES_RESIDENCIAL
2019,3,COMUNICACION CELULAR S A COMCEL S A,15238,DUITAMA,CORPORATIVO,CABLE,300.0,10.0,5,CABLE/HFC,0
2020,4,COMUNICACION CELULAR S A COMCEL S A,15759,SOGAMOSO,RESIDENCIAL - ESTRATO 3,CABLE,15.0,1.02,44,CABLE/HFC,1
2020,4,L@TINNETWORK SAS,15299,GARAGOA,RESIDENCIAL - ESTRATO 2,FIBER TO THE HOME (FTTH),8.0,8.0,4,FIBRA,1
2020,1,ACCESS DIGITAL S.A.S,15835,TURMEQUÉ,CORPORATIVO,WIFI,4.0,1.2,1,INALÁMBRICO,0
2017,4,FUTURE SOLUTIONS DEVELOPMENT S.A.S.,15542,PESCA,RESIDENCIAL - ESTRATO 2,OTRAS TECNOLOGÍAS INALÁMBRICAS,4.0,4.0,6,INALÁMBRICO,1


In [0]:
# Resultados Saber 11 (ICFES)
dfSaber = spark.read.csv(
    "/Volumes/workspace/proyecto/proyectopdd/7__Resultados_Únicos_Saber_11.csv",
    header=True, inferSchema=True
)
dfSaber.createOrReplaceTempView("dfSaber")
print(f"Registros: {dfSaber.count()} | Columnas: {len(dfSaber.columns)}")
display(dfSaber.limit(5))

Registros: 129065 | Columnas: 53


PERIODO,ESTU_TIPODOCUMENTO,ESTU_CONSECUTIVO,COLE_AREA_UBICACION,COLE_BILINGUE,COLE_CALENDARIO,COLE_CARACTER,COLE_COD_DANE_ESTABLECIMIENTO,COLE_COD_DANE_SEDE,COLE_COD_DEPTO_UBICACION,COLE_COD_MCPIO_UBICACION,COLE_CODIGO_ICFES,COLE_DEPTO_UBICACION,COLE_GENERO,COLE_JORNADA,COLE_MCPIO_UBICACION,COLE_NATURALEZA,COLE_NOMBRE_ESTABLECIMIENTO,COLE_NOMBRE_SEDE,COLE_SEDE_PRINCIPAL,ESTU_COD_DEPTO_PRESENTACION,ESTU_COD_MCPIO_PRESENTACION,ESTU_COD_RESIDE_DEPTO,ESTU_COD_RESIDE_MCPIO,ESTU_DEPTO_PRESENTACION,ESTU_DEPTO_RESIDE,ESTU_ESTADOINVESTIGACION,ESTU_ESTUDIANTE,ESTU_FECHANACIMIENTO,ESTU_GENERO,ESTU_MCPIO_PRESENTACION,ESTU_MCPIO_RESIDE,ESTU_NACIONALIDAD,ESTU_PAIS_RESIDE,ESTU_PRIVADO_LIBERTAD,FAMI_CUARTOSHOGAR,FAMI_EDUCACIONMADRE,FAMI_EDUCACIONPADRE,FAMI_ESTRATOVIVIENDA,FAMI_PERSONASHOGAR,FAMI_TIENEAUTOMOVIL,FAMI_TIENECOMPUTADOR,FAMI_TIENEINTERNET,FAMI_TIENELAVADORA,DESEMP_INGLES,PUNT_INGLES,PUNT_MATEMATICAS,PUNT_SOCIALES_CIUDADANAS,PUNT_C_NATURALES,PUNT_LECTURA_CRITICA,PUNT_GLOBAL,NIVEL_DESEMPENO,ACCESO_TECNOLOGICO
20172,TI,SB11201720316876,RURAL,N,A,TÉCNICO,115180000013,115180000013,15,15180,5033,BOYACA,MIXTO,COMPLETA,CHISCAS,OFICIAL,I.E.T. AGROPECUARIO,I.E.T. AGROPECUARIO - SEDE PRINCIPAL,S,15,15244,15,15180,BOYACA,BOYACA,PUBLICAR,ESTUDIANTE,2000-11-13,F,EL COCUY,CHISCAS,COLOMBIA,COLOMBIA,N,Uno,Secundaria (Bachillerato) completa,Primaria completa,Estrato 1,3 a 4,No,No,No,Si,A-,45.0,51.0,45,48,49,240,MEDIO,LIMITADO
20162,TI,SB11201620452664,URBANO,N,A,TÉCNICO/ACADÉMICO,115861000011,115861000011,15,15861,5736,BOYACA,MIXTO,COMPLETA,VENTAQUEMADA,OFICIAL,I.E. FRANCISCO DE PAULA SANTANDER,COL NACIONALIZADO,S,15,15861,15,15861,BOYACA,BOYACA,PUBLICAR,ESTUDIANTE,1999-07-13,F,VENTAQUEMADA,VENTAQUEMADA,COLOMBIA,COLOMBIA,N,Dos,Primaria incompleta,Primaria completa,Estrato 2,Cinco,No,Si,No,Si,A-,46.0,59.0,66,59,59,297,MEDIO,LIMITADO
20162,TI,SB11201620329125,URBANO,N,A,TÉCNICO/ACADÉMICO,115001000367,115001000367,15,15001,4846,BOYACA,MIXTO,MAÑANA,TUNJA,OFICIAL,INSTITUTO DE EDUCACION MEDIA DIVERSIFICADA INEM CARLOS ARTURO TORRES,CARLOS ARTURO TORRES SEDE CENTRAL,S,15,15001,15,15001,BOYACA,BOYACA,PUBLICAR,ESTUDIANTE,1999-12-05,M,TUNJA,TUNJA,COLOMBIA,COLOMBIA,N,Cuatro,Secundaria (Bachillerato) completa,Primaria completa,Estrato 4,Cuatro,No,Si,Si,Si,A1,49.0,65.0,62,55,60,299,MEDIO,ALTO
20162,TI,SB11201620538998,RURAL,N,A,TÉCNICO/ACADÉMICO,115837000256,215837000307,15,15837,175182,BOYACA,MIXTO,MAÑANA,TUTA,OFICIAL,I.E. TÉCNICA CHICAMOCHA,I.E. GENERAL SANTANDER,N,15,15837,15,15837,BOYACA,BOYACA,PUBLICAR,ESTUDIANTE,2000-04-17,F,TUTA,TUTA,COLOMBIA,COLOMBIA,N,Dos,Primaria completa,Secundaria (Bachillerato) incompleta,Estrato 2,Tres,Si,No,No,No,A-,43.0,38.0,46,41,51,219,MEDIO,LIMITADO
20194,CC,SB11201940539754,URBANO,N,A,TÉCNICO/ACADÉMICO,115001002807,115001002807,15,15001,724351,BOYACA,MIXTO,UNICA,TUNJA,OFICIAL,GUSTAVO ROJAS PINILLA.,INSTITUCION EDUCATIVA GUSTAVO ROJAS PINILLA CENTRAL,S,15,15001,15,15001,BOYACA,BOYACA,PUBLICAR,ESTUDIANTE,2000-12-19,M,TUNJA,TUNJA,COLOMBIA,COLOMBIA,N,Dos,Secundaria (Bachillerato) incompleta,Secundaria (Bachillerato) completa,Estrato 2,3 a 4,No,Si,Si,Si,B1,68.0,66.0,68,56,70,326,ALTO,ALTO


## 2. Filtros y Transformaciones

Se realizan las transformaciones finales para estandarizar los datos y construir un dataset maestro agregado por municipio, 
alineado con el objetivo de negocio: analizar la relación entre conectividad a internet y resultados ICFES Saber 11.

### Transformación #0 — Estandarización de nombres de municipios
Se normalizan los nombres de municipio a mayúsculas y sin tildes para garantizar joins correctos entre datasets.


In [0]:
def std_municipio(df, col_name):
    df = df.withColumn("municipio_std", upper(trim(F.col(col_name))))
    df = df.withColumn("municipio_std", translate(F.col("municipio_std"), "ÁÉÍÓÚÄËÏÖÜÑ", "AEIOUAEIOUN"))
    return df

dfInternetOficiales   = std_municipio(dfInternetOficiales,   "municipio")
dfInternetNoOficiales = std_municipio(dfInternetNoOficiales, "municipio")
dfMatricula           = std_municipio(dfMatricula,           "MUNICIPIO")
dfEstadisticas        = std_municipio(dfEstadisticas,        "MUNICIPIO")
dfUNIDOS              = std_municipio(dfUNIDOS,              "NombreMunicipioAtencion")
dfInternet            = std_municipio(dfInternet,            "MUNICIPIO")
dfSaber               = std_municipio(dfSaber,               "COLE_MCPIO_UBICACION")

print("Estandarización completada en todos los DataFrames.")

Estandarización completada en todos los DataFrames.


### Filtro #1 — Eliminación de registros con municipio genérico en Saber 11
Se observó que algunos registros en dfSaber tienen como municipio el nombre del departamento ("BOYACA"), lo que no permite una 
identificación geográfica precisa. Estos registros se eliminan porque no aportan información a nivel municipal.


In [0]:
before = dfSaber.count()
dfSaber = dfSaber.filter(F.col("municipio_std") != "BOYACA")
after  = dfSaber.count()
print(f"Registros eliminados (municipio = BOYACA): {before - after}")
print(f"Registros restantes: {after}")

Registros eliminados (municipio = BOYACA): 422
Registros restantes: 128643


### Filtro #2 — Eliminar outliers extremos de velocidad de internet (IQR)
Se eliminan registros donde la velocidad de bajada sea un valor atípico extremo (por encima del percentil 99),
ya que representan contratos corporativos de altísimo ancho de banda que distorsionarían los promedios municipales.


In [0]:
# Castear a double limpiando posibles strings no numéricos
dfInternet = dfInternet.withColumn(
    "VELOCIDAD_BAJADA",
    F.when(
        (F.col("VELOCIDAD_BAJADA").isNull()) |
        (F.trim(F.col("VELOCIDAD_BAJADA").cast("string")) == "") |
        (F.trim(F.col("VELOCIDAD_BAJADA").cast("string")) == "null"),
        None
    ).otherwise(F.col("VELOCIDAD_BAJADA").cast("double"))
).withColumn(
    "VELOCIDAD_SUBIDA",
    F.when(
        (F.col("VELOCIDAD_SUBIDA").isNull()) |
        (F.trim(F.col("VELOCIDAD_SUBIDA").cast("string")) == "") |
        (F.trim(F.col("VELOCIDAD_SUBIDA").cast("string")) == "null"),
        None
    ).otherwise(F.col("VELOCIDAD_SUBIDA").cast("double"))
).withColumn(
    "No DE ACCESOS",
    F.when(
        (F.col("No DE ACCESOS").isNull()) |
        (F.trim(F.col("No DE ACCESOS").cast("string")) == ""),
        None
    ).otherwise(F.col("No DE ACCESOS").cast("long"))
)

# Verificar tipos
dfInternet.select("VELOCIDAD_BAJADA","VELOCIDAD_SUBIDA","No DE ACCESOS").printSchema()

# Ahora sí calcular percentil 99
p99 = dfInternet.approxQuantile("VELOCIDAD_BAJADA", [0.99], 0.01)[0]
print(f"Percentil 99 velocidad bajada: {p99} Mbps")

before = dfInternet.count()
dfInternet = dfInternet.filter(
    (F.col("VELOCIDAD_BAJADA").isNull()) | (F.col("VELOCIDAD_BAJADA") <= p99)
)
after = dfInternet.count()
print(f"Registros eliminados (outliers velocidad): {before - after}")
print(f"Registros restantes: {after}")

root
 |-- VELOCIDAD_BAJADA: double (nullable = true)
 |-- VELOCIDAD_SUBIDA: double (nullable = true)
 |-- No DE ACCESOS: long (nullable = true)

Percentil 99 velocidad bajada: 100000.0 Mbps
Registros eliminados (outliers velocidad): 0
Registros restantes: 141561


### Transformación #1 — Agregación de Internet Oficial por municipio
Se calculan métricas de conectividad por municipio: porcentaje de sedes con internet, velocidad promedio e índice de conectividad normalizado.


In [0]:
# Convertir columnas de ancho de banda a double
dfInternetOficiales = dfInternetOficiales.withColumn(
    "ancho_subida",
    F.when(F.col("ancho-de-banda-de-subida") == "null", None)
     .otherwise(F.col("ancho-de-banda-de-subida")).cast("double")
)
dfInternetOficiales = dfInternetOficiales.withColumn(
    "ancho_bajada",
    F.when(F.col("ancho-de-banda-descarga") == "null", None)
     .otherwise(F.col("ancho-de-banda-descarga")).cast("double")
)
dfInternetOficiales = dfInternetOficiales.withColumn(
    "tiene_ancho_banda", F.col("tiene_ancho_banda").cast("double")
)

# Agregación por municipio
dfOficialesAGG = dfInternetOficiales.groupBy("municipio_std").agg(
    F.avg("ancho_subida").alias("avg_subida_oficial"),
    F.avg("ancho_bajada").alias("avg_bajada_oficial"),
    F.count("*").alias("total_sedes_oficiales"),
    F.sum("tiene_ancho_banda").alias("sedes_internet_oficiales")
)

w = Window.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

dfOficialesAGG = dfOficialesAGG     .withColumn("pct_internet_oficial",
                F.col("sedes_internet_oficiales") / F.col("total_sedes_oficiales"))     .withColumn("avg_velocidad_oficial",
                (F.col("avg_subida_oficial") + F.col("avg_bajada_oficial")) / 2)     .withColumn("vel_oficial_norm",
                (F.col("avg_velocidad_oficial") - F.min("avg_velocidad_oficial").over(w)) /
                (F.max("avg_velocidad_oficial").over(w) - F.min("avg_velocidad_oficial").over(w)))     .withColumn("indice_conectividad_oficial",
                F.col("vel_oficial_norm") * F.col("pct_internet_oficial"))

dfOficialesAGG = dfOficialesAGG.fillna(0, subset=["vel_oficial_norm", "indice_conectividad_oficial"])
print(f"Municipios en InternetOficiales AGG: {dfOficialesAGG.count()}")
display(dfOficialesAGG.orderBy("pct_internet_oficial").limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Municipios en InternetOficiales AGG: 122


municipio_std,avg_subida_oficial,avg_bajada_oficial,total_sedes_oficiales,sedes_internet_oficiales,pct_internet_oficial,avg_velocidad_oficial,vel_oficial_norm,indice_conectividad_oficial
COPER,5.333333333333333,13.8,24,3.0,0.125,9.566666666666666,0.5295880149812733,0.06619850187265916
PAEZ,5.333333333333333,14.8,23,3.0,0.13043478260869565,10.066666666666666,0.7093632958801497,0.09252564728871518
CHIQUINQUIRA,3.375,18.05,36,6.0,0.16666666666666666,10.7125,0.9415730337078653,0.15692883895131088
MIRAFLORES,5.3125,15.833333333333334,22,4.0,0.18181818181818182,10.572916666666668,0.8913857677902626,0.16207013959822955
SAN LUIS DE GACENO,4.55,14.555555555555555,25,5.0,0.2,9.552777777777777,0.5245942571785266,0.10491885143570533
BUENAVISTA,3.1875,15.166666666666666,20,4.0,0.2,9.177083333333332,0.3895131086142318,0.07790262172284636
BRICENO,3.0,15.2,14,3.0,0.21428571428571427,9.1,0.3617977528089886,0.07752808988764041
SAN MATEO,4.7,14.0,23,5.0,0.21739130434782608,9.35,0.4516853932584268,0.09819247679531018
PAUNA,3.28125,14.545454545454545,36,8.0,0.2222222222222222,8.913352272727273,0.29468845760980616,0.06548632391329026
QUIPAMA,4.291666666666667,14.222222222222221,26,6.0,0.23076923076923078,9.256944444444445,0.41822721598002505,0.09651397291846732


### Transformación #2 — Agregación de Internet No Oficial por municipio
Se aplica la misma lógica al sector privado para comparar cobertura y velocidad entre sectores.


In [0]:
dfInternetNoOficiales = dfInternetNoOficiales.withColumn(
    "ancho_banda",
    F.when((F.col("ancho-de-banda-num") == "null") | (F.col("ancho-de-banda-num") == ""), None)
     .otherwise(F.col("ancho-de-banda-num").cast("double"))
)
dfInternetNoOficiales = dfInternetNoOficiales.withColumn("velocidad", F.col("ancho_banda"))
dfInternetNoOficiales = dfInternetNoOficiales.withColumn(
    "tiene_ancho_banda_num", F.col("tiene-ancho-banda").cast("double")
)

dfNoOficialesAGG = dfInternetNoOficiales.groupBy("municipio_std").agg(
    F.avg("velocidad").alias("avg_velocidad_privada"),
    F.count("*").alias("total_sedes_privadas"),
    F.sum("tiene_ancho_banda_num").alias("sedes_internet_privadas")
)

dfNoOficialesAGG = dfNoOficialesAGG     .withColumn("pct_internet_privado",
                F.col("sedes_internet_privadas") / F.col("total_sedes_privadas"))     .withColumn("vel_privada_norm",
                (F.col("avg_velocidad_privada") - F.min("avg_velocidad_privada").over(w)) /
                (F.max("avg_velocidad_privada").over(w) - F.min("avg_velocidad_privada").over(w)))     .withColumn("indice_conectividad_privado",
                F.col("vel_privada_norm") * F.col("pct_internet_privado"))

dfNoOficialesAGG = dfNoOficialesAGG.fillna(0, subset=["vel_privada_norm","indice_conectividad_privado"])
print(f"Municipios en InternetNoOficiales AGG: {dfNoOficialesAGG.count()}")
display(dfNoOficialesAGG.orderBy("pct_internet_privado").limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Municipios en InternetNoOficiales AGG: 123


municipio_std,avg_velocidad_privada,total_sedes_privadas,sedes_internet_privadas,pct_internet_privado,vel_privada_norm,indice_conectividad_privado
ALMEIDA,null,1,0.0,0.0,0.0,0.0
SOGAMOSO,null,5,0.0,0.0,0.0,0.0
TUNJA,null,3,0.0,0.0,0.0,0.0
TOTA,15.0,13,1.0,0.07692307692307693,0.5,0.038461538461538464
SATIVANORTE,12.5,8,2.0,0.25,0.25,0.0625
NOBSA,17.5,8,2.0,0.25,0.75,0.1875
TIBASOSA,11.666666666666666,10,3.0,0.3,0.1666666666666666,0.04999999999999998
UMBITA,11.25,12,4.0,0.3333333333333333,0.125,0.041666666666666664
OICATA,15.0,3,1.0,0.3333333333333333,0.5,0.16666666666666666
PAJARITO,15.0,3,1.0,0.3333333333333333,0.5,0.16666666666666666


### Transformación #3 — Agregación de Saber 11 (puntaje promedio por municipio)
Se calcula el puntaje global promedio en el Saber 11 por municipio, que es la variable objetivo principal.


In [0]:
dfSaberAGG = dfSaber.groupBy("municipio_std").agg(
    F.avg("PUNT_GLOBAL").alias("avg_punt_global"),
    F.avg("PUNT_MATEMATICAS").alias("avg_matematicas"),
    F.avg("PUNT_LECTURA_CRITICA").alias("avg_lectura"),
    F.avg("PUNT_SOCIALES_CIUDADANAS").alias("avg_sociales"),
    F.avg("PUNT_C_NATURALES").alias("avg_ciencias"),  # <--- Columna corregida
    F.avg("PUNT_INGLES").alias("avg_ingles"),
    F.count("*").alias("total_estudiantes_saber")
)
print(f"Municipios en SaberAGG: {dfSaberAGG.count()}")
display(dfSaberAGG.orderBy(F.desc("avg_punt_global")).limit(10))

Municipios en SaberAGG: 123


municipio_std,avg_punt_global,avg_matematicas,avg_lectura,avg_sociales,avg_ciencias,avg_ingles,total_estudiantes_saber
PAIPA,287.43236227285837,59.94779348139602,57.456013844822614,55.591577732910295,57.80011537352178,54.913758292471876,3467
DUITAMA,281.4103055875208,57.99041043344841,56.86254954609385,54.87258662575118,55.77873673443294,55.134317862165965,15642
TUNJA,280.7382045521149,57.38595291589333,56.99360164110579,54.68530819576048,55.44627332226238,56.39484223893719,20474
SOGAMOSO,278.51059391094594,56.77560466784015,56.66647108677228,54.255883695156136,55.27531129799856,55.19929591238021,15339
SANTA ROSA DE VITERBO,277.3871951219512,57.75609756097561,55.675813008130085,54.25813008130081,54.96544715447155,53.17479674796748,984
NOBSA,275.75,56.78861256544503,55.909685863874344,53.11060209424084,54.7565445026178,55.25458115183246,1528
IZA,274.9770992366412,58.6793893129771,54.954198473282446,52.98473282442748,54.85496183206107,50.56488549618321,131
COMBITA,273.3169897377423,56.60091220068415,54.52223489167617,53.01482326111745,54.84720638540479,53.69897377423033,877
SAN EDUARDO,270.1904761904762,55.55952380952381,53.42261904761905,52.458333333333336,55.36904761904762,51.99404761904762,168
CERINZA,269.9865229110512,55.67385444743935,54.38544474393531,52.3800539083558,54.82749326145552,50.16711590296496,371


### Transformación #4 — Agregación Internet Fijo Residencial por municipio
Se extrae la cobertura de internet fijo residencial como proxy del acceso a internet en los hogares.


In [0]:
# residencial 1/0
dfInternet = dfInternet.withColumn(
    "ES_RESIDENCIAL",
    F.when(F.col("ES_RESIDENCIAL").cast("string").isin("1","1.0"), 1).otherwise(0)
)

#eliminar registros sin suscriptores
dfInternet = dfInternet.withColumn(
    "NUM_ACCESOS", F.col("`No DE ACCESOS`").cast("integer")
)
dfInternet = dfInternet.filter(F.col("NUM_ACCESOS") > 0)
dfInternet = dfInternet.filter(
    F.col("SEGMENTO") != "USO PROPIO INTERNO DEL OPERADOR"
)

dfInternet = dfInternet \
    .withColumn("VELOCIDAD_BAJADA",
        F.when(F.col("VELOCIDAD_BAJADA") == 0, None).otherwise(F.col("VELOCIDAD_BAJADA"))) \
    .withColumn("VELOCIDAD_SUBIDA",
        F.when(F.col("VELOCIDAD_SUBIDA") == 0, None).otherwise(F.col("VELOCIDAD_SUBIDA")))

# tipo de tech.
dfInternet = dfInternet.withColumn(
    "TIPO_TECNOLOGIA",
    F.when(F.col("TECNOLOGIA").contains("FIBER") | F.col("TECNOLOGIA").contains("FTTX"), "FIBRA")
     .when(F.col("TECNOLOGIA").isin("CABLE", "HYBRID FIBER COAXIAL (HFC)"), "CABLE/HFC")
     .when(F.col("TECNOLOGIA") == "XDSL", "XDSL")
     .when(F.col("TECNOLOGIA") == "SATELITAL", "SATELITAL")
     .when(F.col("TECNOLOGIA").isin("WIFI", "WIMAX", "OTRAS TECNOLOGÍAS INALÁMBRICAS"), "INALÁMBRICO")
     .otherwise("OTRAS")
)

w_global = Window.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

# velocidad promedio ponderada
df_internet_vel = dfInternet.groupBy("municipio_std").agg(
    (F.sum(F.col("VELOCIDAD_BAJADA") * F.col("NUM_ACCESOS")) /
     F.sum("NUM_ACCESOS")).alias("avg_vel_bajada"),
    (F.sum(F.col("VELOCIDAD_SUBIDA") * F.col("NUM_ACCESOS")) /
     F.sum("NUM_ACCESOS")).alias("avg_vel_subida"),
    F.sum("NUM_ACCESOS").alias("total_accesos_fijo"),
    F.count("*").alias("registros_internet")
)
df_internet_vel = df_internet_vel \
    .withColumn("avg_velocidad_fijo",
        (F.col("avg_vel_bajada") + F.col("avg_vel_subida")) / 2) \
    .withColumn("velocidad_norm_fijo",
        (F.col("avg_velocidad_fijo") - F.min("avg_velocidad_fijo").over(w_global)) /
        (F.max("avg_velocidad_fijo").over(w_global) - F.min("avg_velocidad_fijo").over(w_global)))

# % accesos
df_internet_res = dfInternet.groupBy("municipio_std").agg(
    (F.sum(F.when(F.col("ES_RESIDENCIAL") == 1, F.col("NUM_ACCESOS")).otherwise(0)) /
     F.sum("NUM_ACCESOS")).alias("pct_accesos_residencial")
)

# tecnología predominante
df_tech = (
    dfInternet
    .groupBy("municipio_std", "TIPO_TECNOLOGIA")
    .agg(F.sum("NUM_ACCESOS").alias("accesos_por_tech"))
    .withColumn("rank",
        F.row_number().over(
            Window.partitionBy("municipio_std").orderBy(F.col("accesos_por_tech").desc())
        )
    )
    .filter(F.col("rank") == 1)
    .select("municipio_std", F.col("TIPO_TECNOLOGIA").alias("tecnologia_predominante"))
)

# join
dfInternetFijoAGG = (
    df_internet_vel
    .join(df_internet_res, on="municipio_std", how="left")
    .join(df_tech,         on="municipio_std", how="left")
)
dfInternetFijoAGG = dfInternetFijoAGG.withColumn(
    "acceso_digital_residencial",
    F.coalesce(F.col("pct_accesos_residencial"), F.lit(0.0)) *
    F.coalesce(F.col("velocidad_norm_fijo"),      F.lit(0.0))
)

print(f"Municipios en InternetFijoAGG: {dfInternetFijoAGG.count()}")
display(dfInternetFijoAGG.orderBy(F.desc("total_accesos_fijo")).limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Municipios en InternetFijoAGG: 123


municipio_std,avg_vel_bajada,avg_vel_subida,total_accesos_fijo,registros_internet,avg_velocidad_fijo,velocidad_norm_fijo,pct_accesos_residencial,tecnologia_predominante,acceso_digital_residencial
TUNJA,148.75860460052232,42.57079188266253,1184822,22930,95.66469824159242,0.14951045192504595,0.9131236590812797,CABLE/HFC,0.1365215309326937
SOGAMOSO,96.33109472221943,18.86274516513999,689627,15105,57.59691994367971,0.08762032315486537,0.8894184827450201,CABLE/HFC,0.07793113487802872
DUITAMA,112.38835596094547,22.833351990370407,678743,15574,67.61085397565795,0.10390085473422495,0.8957175248952843,CABLE/HFC,0.09306581643704445
PUERTO BOYACA,19.9790951772885,10.15863289865728,155058,4532,15.06886403797289,0.018478729554328086,0.816429981039353,FIBRA,0.015086588819671412
CHIQUINQUIRA,31.587663945405584,17.051982925144173,141846,5859,24.319823435274877,0.033518826296147275,0.8636760994317781,XDSL,0.028949409152987795
PAIPA,20.150282849749175,7.886489569328185,138342,5667,14.01838620953868,0.016770875537652313,0.9360642465773229,FIBRA,0.015698616974594567
VILLA DE LEYVA,50.748231429853234,40.321551304236365,62412,4723,45.5348913670448,0.06801002449242767,0.7792732166890982,XDSL,0.05299839055331847
NOBSA,17.52382491353405,13.711290468900607,35274,2762,15.617557691217328,0.019370788990760734,0.8820944605091569,FIBRA,0.017086865664441803
MONIQUIRA,16.329704980283335,15.161388199211332,27388,1832,15.745546589747335,0.019578871777972143,0.9039360303782679,FIBRA,0.01769804763426524
TIBASOSA,26.65770072707372,24.211870846689077,25582,1584,25.4347857868814,0.03533151846259857,0.8745993276522555,FIBRA,0.03090092229232196


### Transformación #5 — Agregación UNIDOS (índice de pobreza y logros sociales por municipio)
Se calcula el puntaje SISBEN promedio y el promedio de logros alcanzados como proxy de pobreza y capital social.


In [0]:
dfUNIDOS_clean = dfUNIDOS \
    .filter(F.col("EstadoBeneficiario") == "ACTIVO") \
    .filter(F.col("Estrato") != "99")

# Reemplazar "ND" por null
for c in ["Genero", "Parentesco", "Discapacidad", "EstadoCivil"]:
    dfUNIDOS_clean = dfUNIDOS_clean.withColumn(
        c, F.when(F.col(c) == "ND", None).otherwise(F.col(c))
    )

# total_logros_alcanzados
logros = [f"Logro{i}" for i in range(1, 27)]
dfUNIDOS_clean = dfUNIDOS_clean.withColumn(
    "total_logros_alcanzados",
    sum([F.when(F.col(l) == "ALCANZADO", 1).otherwise(0) for l in logros])
)

# nivel_educativo_bajo 0/1
niveles_bajos = [
    "BÁSICA PRIMARIA 1°", "BÁSICA PRIMARIA 2°", "BÁSICA PRIMARIA 3°",
    "BÁSICA PRIMARIA 4°", "BÁSICA PRIMARIA 5°", "NINGUNO"
]
dfUNIDOS_clean = dfUNIDOS_clean.withColumn(
    "nivel_educativo_bajo",
    F.when(F.col("PE42").isin(niveles_bajos), 1).otherwise(0)
)

# AGG por municipio
dfUNIDOSAGG = dfUNIDOS_clean.groupBy("municipio_std").agg(
    F.count("*").alias("hogares_unidos"),
    F.avg("total_logros_alcanzados").alias("avg_logros_alcanzados"),
    (F.sum("nivel_educativo_bajo") / F.count("*")).alias("pct_nivel_educativo_bajo"),
    (F.count(F.when(F.col("TipoPoblacion") == "UNIDOS RURAL", 1)) / F.count("*"))
        .alias("pct_rural_unidos"),
    F.avg(F.col("Estrato").cast("double")).alias("avg_estrato_unidos")
)

# Normalización
w_global = Window.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
dfUNIDOSAGG = dfUNIDOSAGG.withColumn(
    "hogares_unidos_norm",
    (F.col("hogares_unidos") - F.min("hogares_unidos").over(w_global)) /
    (F.max("hogares_unidos").over(w_global) - F.min("hogares_unidos").over(w_global))
)

print(f"Municipios en UNIDOS AGG: {dfUNIDOSAGG.count()}")
display(dfUNIDOSAGG.orderBy(F.desc("hogares_unidos")).limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Municipios en UNIDOS AGG: 122


municipio_std,hogares_unidos,avg_logros_alcanzados,pct_nivel_educativo_bajo,pct_rural_unidos,avg_estrato_unidos,hogares_unidos_norm
CHIQUINQUIRA,1637,10.301160659743433,0.5400122174709835,0.13072693952351863,0.08063530849114234,1.0
TUNJA,1557,10.255619781631342,0.49582530507385997,0.038535645472061654,0.09120102761721259,0.9510703363914373
CHITA,1253,6.264964086193136,0.707901037509976,1.0,0.08140462889066241,0.7651376146788991
SOGAMOSO,1224,9.031862745098039,0.5686274509803921,0.6160130718954249,0.10784313725490197,0.7474006116207951
COMBITA,1115,9.531838565022422,0.5874439461883408,0.9973094170403587,0.10762331838565023,0.6807339449541284
PAUNA,1109,9.098286744815148,0.6807935076645627,1.0,0.1406672678088368,0.6770642201834862
MARIPI,1049,8.612011439466158,0.7626310772163966,0.998093422306959,0.21258341277407056,0.6403669724770642
SABOYA,1024,9.265625,0.6171875,0.9951171875,0.0654296875,0.6250764525993884
SAMACA,993,9.715005035246728,0.6243705941591138,0.7653575025176234,0.06445115810674723,0.6061162079510704
CHIQUIZA,850,8.987058823529411,0.7035294117647058,0.9964705882352941,0.1376470588235294,0.5186544342507645


### Transformación #6 — Estadísticas Educativas: promedio de coberturas y deserción por municipio
Se toman los datos de cobertura neta (especialmente secundaria y media) y deserción, que son indicadores clave.


In [0]:
# ============================
# LIMPIEZA Y AGREGACIÓN DE dfEstadisticas
# ============================

import pyspark.sql.functions as F

# 1. Ver el estado original (solo informativo)
dfEstadisticas.select(
    "municipio_std",
    "COBERTURA_NETA_SECUNDARIA",
    "COBERTURA_NETA_MEDIA",
    "DESERCIÓN_SECUNDARIA",
    "APROBACIÓN_SECUNDARIA",
    "SEDES_CONECTADAS_A_INTERNET"
).show(10, False)

dfEstadisticas.select(
    "COBERTURA_NETA_SECUNDARIA",
    "COBERTURA_NETA_MEDIA",
    "DESERCIÓN_SECUNDARIA",
    "APROBACIÓN_SECUNDARIA",
    "SEDES_CONECTADAS_A_INTERNET"
).printSchema()

# 2. Función robusta para porcentajes con comas y %
def clean_pct_safe(df, colname):
    """
    Limpia una columna de porcentaje tipo '98,71%' o '6,25%':
      - Trata 'null', '' y espacios como NULL reales
      - Quita '%' y comas
      - Castea a double
      - Crea <colname>_num
    """
    c_str = F.col(colname).cast("string")
    return df.withColumn(
        colname + "_num",
        F.when(
            c_str.isNull() |
            (F.trim(c_str) == "") |
            (F.trim(c_str) == "null"),
            None
        ).otherwise(
            F.regexp_replace(c_str, "[%,]", "").cast("double")
        )
    )

# 3. Aplicar a las cuatro columnas de porcentaje
for c in [
    "COBERTURA_NETA_SECUNDARIA",
    "COBERTURA_NETA_MEDIA",
    "DESERCIÓN_SECUNDARIA",
    "APROBACIÓN_SECUNDARIA"
]:
    dfEstadisticas = clean_pct_safe(dfEstadisticas, c)

# 4. Asegurar que SEDES_CONECTADAS_A_INTERNET también sea numérica segura
c_str = F.col("SEDES_CONECTADAS_A_INTERNET").cast("string")
dfEstadisticas = dfEstadisticas.withColumn(
    "SEDES_CONECTADAS_A_INTERNET",
    F.when(
        c_str.isNull() |
        (F.trim(c_str) == "") |
        (F.trim(c_str) == "null"),
        None
    ).otherwise(
        F.regexp_replace(c_str, "[%,]", "").cast("double")
    )
)

# 5. Verificar que ya todo quede en double
dfEstadisticas.select(
    "COBERTURA_NETA_SECUNDARIA_num",
    "COBERTURA_NETA_MEDIA_num",
    "DESERCIÓN_SECUNDARIA_num",
    "APROBACIÓN_SECUNDARIA_num",
    "SEDES_CONECTADAS_A_INTERNET"
).printSchema()

dfEstadisticas.select(
    "COBERTURA_NETA_SECUNDARIA_num",
    "COBERTURA_NETA_MEDIA_num",
    "DESERCIÓN_SECUNDARIA_num",
    "APROBACIÓN_SECUNDARIA_num",
    "SEDES_CONECTADAS_A_INTERNET"
).show(10, False)

# 6. Agregación sin errores de cast
dfEstadisticasAGG = dfEstadisticas.groupBy("municipio_std").agg(
    F.avg("COBERTURA_NETA_SECUNDARIA_num").alias("avg_cobertura_secundaria"),
    F.avg("COBERTURA_NETA_MEDIA_num").alias("avg_cobertura_media"),
    F.avg("DESERCIÓN_SECUNDARIA_num").alias("avg_desercion_secundaria"),
    F.avg("APROBACIÓN_SECUNDARIA_num").alias("avg_aprobacion_secundaria"),
    F.avg("SEDES_CONECTADAS_A_INTERNET").alias("avg_sedes_conectadas")
)

print(f"Municipios en EstadisticasAGG: {dfEstadisticasAGG.count()}")
display(dfEstadisticasAGG.limit(10))

+-------------+-------------------------+--------------------+--------------------+---------------------+---------------------------+
|municipio_std|COBERTURA_NETA_SECUNDARIA|COBERTURA_NETA_MEDIA|DESERCIÓN_SECUNDARIA|APROBACIÓN_SECUNDARIA|SEDES_CONECTADAS_A_INTERNET|
+-------------+-------------------------+--------------------+--------------------+---------------------+---------------------------+
|NOBSA        |98,71%                   |60,8%               |4,08%               |79,31%               |null                       |
|CORRALES     |85,82%                   |48,65%              |2,19%               |89,78%               |null                       |
|COVARACHIA   |63,48%                   |53,01%              |4,08%               |90,48%               |null                       |
|SUSACON      |61,29%                   |57,69%              |2,86%               |97,14%               |null                       |
|MACANAL      |58,99%                   |55%                 |

municipio_std,avg_cobertura_secundaria,avg_cobertura_media,avg_desercion_secundaria,avg_aprobacion_secundaria,avg_sedes_conectadas
NOBSA,4979.5,3623.0,303.7142857142857,7491.214285714285,210.71428571428572
SUSACON,4905.214285714285,3782.3571428571427,130.14285714285714,8363.857142857143,1964.4285714285713
PAEZ,4924.785714285715,4017.6428571428573,306.7142857142857,9249.642857142857,565.2857142857143
MARIPI,4943.928571428572,2338.9285714285716,391.2857142857143,7776.571428571428,2358.0
CAMPOHERMOSO,4348.785714285715,1737.7857142857142,419.85714285714283,6763.857142857143,1253.2857142857142
TOGUI,5835.642857142857,4131.142857142857,660.7142857142857,7838.357142857143,1938.857142857143
IZA,6642.5,2425.6428571428573,466.7857142857143,7881.642857142857,965.2857142857143
CHITA,6081.0,4285.357142857143,288.92857142857144,7673.142857142857,1862.5714285714287
ALMEIDA,3968.785714285714,3164.9285714285716,177.75,7357.857142857143,771.4285714285714
COMBITA,6337.428571428572,3743.5,405.14285714285717,6812.357142857143,1800.857142857143


### Transformación #7 — Matrícula universitaria por municipio (último año disponible)
Se extrae el total de estudiantes universitarios como indicador del nivel de educación superior.


In [0]:
dfMatriculaReciente = dfMatricula.filter(F.col("AÑO") == dfMatricula.select(F.max("AÑO")).first()[0])

dfMatriculaAGG = dfMatriculaReciente.groupBy("municipio_std").agg(
    F.sum("TOTAL_ESTUDIANTES").alias("total_estudiantes_universitarios"),
    F.sum("UNIVERSITARIA").alias("matricula_universitaria"),
    F.count("*").alias("num_registros_matricula")
)
print(f"Municipios en MatriculaAGG: {dfMatriculaAGG.count()}")
display(dfMatriculaAGG.orderBy(F.desc("total_estudiantes_universitarios")).limit(10))

Municipios en MatriculaAGG: 16


municipio_std,total_estudiantes_universitarios,matricula_universitaria,num_registros_matricula
TUNJA,37261,30422,1
SOGAMOSO,15987,6432,1
DUITAMA,10880,5849,1
SANTA ROSA DE VITERBO,9880,0,1
CHIQUINQUIRA,5367,3027,1
GARAGOA,5070,4270,1
CUBARA,4140,3730,1
SOATA,3680,2990,1
PUERTO BOYACA,3520,0,1
SOCHA,2830,2670,1


### JOIN MAESTRO — Construcción del Dataset Consolidado por Municipio
Se unen todos los datasets procesados usando el campo `municipio_std` como llave de cruce. 
Se utiliza `left join` a partir del dataset de Saber 11 para preservar todos los municipios con resultados ICFES.


In [0]:
dfMaestro = dfSaberAGG \
    .join(dfOficialesAGG.select(
        "municipio_std", "pct_internet_oficial", "avg_velocidad_oficial",
        "indice_conectividad_oficial", "total_sedes_oficiales"),
        on="municipio_std", how="left") \
    .join(dfNoOficialesAGG.select(
        "municipio_std", "pct_internet_privado", "avg_velocidad_privada",
        "indice_conectividad_privado", "total_sedes_privadas"),
        on="municipio_std", how="left") \
    .join(dfInternetFijoAGG.select(
        "municipio_std",
        "total_accesos_fijo",
        "avg_velocidad_fijo",
        "velocidad_norm_fijo",
        "pct_accesos_residencial",
        "tecnologia_predominante",
        "acceso_digital_residencial"),
        on="municipio_std", how="left") \
    .join(dfUNIDOSAGG.select(
        "municipio_std",
        "hogares_unidos",
        "avg_logros_alcanzados",
        "pct_nivel_educativo_bajo",
        "pct_rural_unidos",
        "avg_estrato_unidos",
        "hogares_unidos_norm"),
        on="municipio_std", how="left") \
    .join(dfEstadisticasAGG, on="municipio_std", how="left") \
    .join(dfMatriculaAGG,    on="municipio_std", how="left")

print("Municipios en dataset maestro:", dfMaestro.count())
print("Columnas totales:", len(dfMaestro.columns))
display(dfMaestro.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Municipios en dataset maestro: 123
Columnas totales: 36


municipio_std,avg_punt_global,avg_matematicas,avg_lectura,avg_sociales,avg_ciencias,avg_ingles,total_estudiantes_saber,pct_internet_oficial,avg_velocidad_oficial,indice_conectividad_oficial,total_sedes_oficiales,pct_internet_privado,avg_velocidad_privada,indice_conectividad_privado,total_sedes_privadas,total_accesos_fijo,avg_velocidad_fijo,velocidad_norm_fijo,pct_accesos_residencial,tecnologia_predominante,acceso_digital_residencial,hogares_unidos,avg_logros_alcanzados,pct_nivel_educativo_bajo,pct_rural_unidos,avg_estrato_unidos,hogares_unidos_norm,avg_cobertura_secundaria,avg_cobertura_media,avg_desercion_secundaria,avg_aprobacion_secundaria,avg_sedes_conectadas,total_estudiantes_universitarios,matricula_universitaria,num_registros_matricula
GUAYATA,254.55555555555554,52.513888888888886,50.83680555555556,50.420138888888886,50.87847222222222,48.0,288,0.3333333333333333,9.589285714285715,0.1792402354200108,12,0.5,12.5,0.125,4,1923,6.200397815912636,0.004060485542767236,0.38845553822152884,FIBRA,0.001577318096956383,188,9.542553191489361,0.7021276595744681,0.9893617021276596,0.10638297872340426,0.11376146788990826,5151.928571428572,3980.6428571428573,321.7142857142857,7591.785714285715,1264.7142857142858,null,null,null
GUACAMAYAS,256.19879518072287,51.62048192771084,52.295180722891565,50.19277108433735,51.397590361445786,49.75301204819277,166,0.375,8.791666666666666,0.09410112359550554,8,1.0,10.0,0.0,1,435,5.955034482758621,0.003661576834054509,0.6689655172413793,FIBRA,0.0024494686407123267,206,9.970873786407767,0.7233009708737864,1.0,0.3446601941747573,0.12477064220183487,5038.714285714285,3802.4285714285716,243.58333333333334,8363.357142857143,917.5714285714286,null,null,null
JERICO,236.171875,48.984375,48.1,45.2625,47.7125,43.803125,320,0.5789473684210527,8.313811188811188,0.045808124459809745,19,1.0,12.5,0.25,2,1132,8.287694346289754,0.007453986735183138,0.8003533568904594,FIBRA,0.00596582330572078,558,9.14336917562724,0.7508960573476703,0.996415770609319,0.16308243727598568,0.3400611620795107,2860.8571428571427,3379.214285714286,432.7857142857143,8065.785714285715,4135.428571428572,null,null,null
BUSBANZA,255.55263157894737,51.86842105263158,53.05263157894737,48.89473684210526,50.81578947368421,50.86842105263158,38,0.6666666666666666,10.25,0.5168539325842696,3,0.5,10.0,0.0,2,981,5.4957543323139655,0.0029148847778446988,0.8583078491335372,INALÁMBRICO,0.002501868484143972,66,9.484848484848484,0.6060606060606061,1.0,0.2727272727272727,0.03914373088685015,4063.214285714286,2933.6,260.6923076923077,5923.357142857143,50.0,null,null,null
FIRAVITOBA,259.19624217118997,52.659707724425886,52.48016701461378,50.04384133611691,52.58872651356994,50.61169102296451,479,0.4444444444444444,9.177083333333332,0.17311693716188079,9,1.0,13.333333333333334,0.33333333333333337,3,7911,18.37780748325117,0.02385836937251907,0.9340159271899886,FIBRA,0.022284096990714625,297,9.232323232323232,0.6262626262626263,0.936026936026936,0.20202020202020202,0.18042813455657492,5712.785714285715,3822.8571428571427,178.28571428571428,8993.5,2088.5714285714284,null,null,null
CERINZA,269.9865229110512,55.67385444743935,54.38544474393531,52.3800539083558,54.82749326145552,50.16711590296496,371,0.25,10.5,0.21629213483146068,12,0.5,15.0,0.25,2,4554,5.778745059288537,0.0033749676436257993,0.8416776460254721,INALÁMBRICO,0.002840634821699097,185,9.151351351351352,0.5945945945945946,0.9567567567567568,0.15135135135135136,0.11192660550458716,6008.214285714285,4864.428571428572,133.42857142857142,7461.714285714285,2381.1428571428573,null,null,null
CHINAVITA,253.81690140845072,52.0,51.271126760563384,49.83450704225352,51.313380281690144,46.58450704225352,284,0.45454545454545453,9.489285714285714,0.22807529549102573,11,0.4,12.5,0.1,5,1529,6.569493132766514,0.004660556198579835,0.8181818181818182,FIBRA,0.0038131823442925923,145,9.6,0.6896551724137931,1.0,0.10344827586206896,0.0874617737003058,5670.5,4191.714285714285,536.5,7721.857142857143,3835.8571428571427,null,null,null
TI